# Experiment 003 — Final reconstructed larger/fake-heavy visual clip run


> This is **not** claimed to be the literal historical notebook that executed
> the archived run. The historical Exp003 notebook snapshot was stale and did
> not match the final archived ConvNeXt-Base configuration. This notebook
> reconstructs the final executable pipeline using the authoritative final
> configuration, split manifests, metrics and the shared visual-feature code.


This notebook reconstructs the **final Exp003 configuration** from the
authoritative experiment config/results plus the shared visual preprocessing
pipeline.

Final design:

- AV-Deepfake1M++ usable validation subset
- exact archived Exp003 train/validation/test manifests
- larger clip-group-disjoint, naturally fake-heavy split: 41,305 / 8,818 / 8,953
- frozen ImageNet ConvNeXt-Base
- **64** uniformly sampled frames per clip
- mean-pooled 1024-D visual representation
- `LayerNorm(1024) -> Linear(1024, 2)` classifier
- classifier selected by validation F1
- test operating point = ordinary two-class argmax / probability threshold 0.5


## 1. Imports and reproducibility

The feature extraction stage is deterministic for fixed videos, frame indices,
torchvision weights and preprocessing. The classifier stage is seeded. GPU and
library differences can still cause very small numerical differences, so the
final historical metrics are treated as a comparison target rather than a
bit-for-bit guarantee.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import math
import random
import shutil
import warnings
import zipfile

import cv2
cv2.setNumThreads(0)

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

from torchvision.models import convnext_base, ConvNeXt_Base_Weights

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from IPython.display import display, FileLink

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Match the original project behaviour while keeping the seed explicit.
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Final experiment configuration

In [ ]:
PROJECT_ROOT = Path("/home/jovyan/MSC_PROJECT")

EXPERIMENT_ID = "exp003"
RUN_NAME = "exp003_visual_clip_final_reproduction"

PATH_COL = "path"
LABEL_COL = "binary_label"
CONDITION_COL = "condition"
GROUP_COL = "clip_group"

MODEL_NAME = "convnext_base"
NUM_FRAMES = 64
IMAGE_SIZE = 224
FEATURE_DIM = 1024

# Final archived feature-extraction settings.
FEATURE_BATCH_SIZE = 32
NUM_WORKERS = 2
PREFETCH_FACTOR = 1
PIN_MEMORY = False

# Final archived classifier settings.
CLASSIFIER_BATCH_SIZE = 2048
CLASSIFIER_EPOCHS = 30
CLASSIFIER_LR = 1e-3
CLASSIFIER_WEIGHT_DECAY = 1e-4

EXPECTED_SPLIT_ROWS = {
    "train": 41305,
    "val": 8818,
    "test": 8953,
}

EXPECTED_HISTORICAL_METRICS = {'accuracy': 0.755054, 'balanced_accuracy': 0.683153, 'f1': 0.830329, 'precision': 0.760488, 'recall': 0.914295, 'auc': 0.797068}

# Set this manually if auto-discovery cannot find the archived manifests.
SPLIT_MANIFEST_DIR = None

RUN_DIR = PROJECT_ROOT / "experiments" / RUN_NAME
MANIFEST_OUT_DIR = RUN_DIR / "manifests"
FEATURE_DIR = RUN_DIR / "features"
RESULT_DIR = RUN_DIR / "results"
PLOT_DIR = RUN_DIR / "plots"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"

for d in [RUN_DIR, MANIFEST_OUT_DIR, FEATURE_DIR, RESULT_DIR, PLOT_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "experiment_id": EXPERIMENT_ID,
    "run_name": RUN_NAME,
    "status": "reconstructed_final_reproduction_notebook",
    "historical_notebook_status": "stale_not_used_as_authoritative_execution_evidence",
    "dataset": "AV-Deepfake1M++ validation subset",
    "task": "visual_only_binary_classification",
    "target": LABEL_COL,
    "split_type": "exact_archived_clip_group_disjoint_manifests",
    "model": "Frozen ConvNeXt-Base encoder + LayerNorm/Linear classifier",
    "num_frames": NUM_FRAMES,
    "image_size": IMAGE_SIZE,
    "feature_dim": FEATURE_DIM,
    "feature_batch_size": FEATURE_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR,
    "pin_memory": PIN_MEMORY,
    "classifier_batch_size": CLASSIFIER_BATCH_SIZE,
    "classifier_epochs": CLASSIFIER_EPOCHS,
    "classifier_lr": CLASSIFIER_LR,
    "classifier_weight_decay": CLASSIFIER_WEIGHT_DECAY,
    "checkpoint_selection": "best validation F1",
    "test_threshold": 0.5,
    "seed": SEED,
}

with open(RUN_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

CONFIG

## 3. Locate and validate the exact final split manifests

**Important:** the split is part of the experiment. A newly generated split,
even if group-disjoint and the same size, would be a new experiment.

The code therefore searches for the saved Exp002/Exp003 manifests and stops if
it cannot find exactly one plausible train, validation and test CSV.

In [ ]:
CANDIDATE_MANIFEST_DIRS = [
    PROJECT_ROOT / "experiments" / "exp003_visual_fullval" / "manifests",
    PROJECT_ROOT / "experiments" / "003_visual_convnext_base_frozen_features_linear_group_disjoint_balancedfull_val_16frames" / "manifests",
    PROJECT_ROOT / "supporting_material" / "experiments" / "exp003_visual_fullval" / "manifests"
]

def choose_manifest_dir():
    if SPLIT_MANIFEST_DIR is not None:
        d = Path(SPLIT_MANIFEST_DIR)
        if not d.is_dir():
            raise FileNotFoundError(f"Configured SPLIT_MANIFEST_DIR does not exist: {d}")
        return d

    existing = [d for d in CANDIDATE_MANIFEST_DIRS if d.is_dir()]
    if len(existing) == 1:
        return existing[0]
    if len(existing) == 0:
        raise FileNotFoundError(
            "Could not find the archived Exp003 manifest directory. "
            "Set SPLIT_MANIFEST_DIR manually to the folder containing the exact "
            "final train/val/test CSV files."
        )
    raise RuntimeError(
        "Multiple candidate manifest directories were found. Set "
        f"SPLIT_MANIFEST_DIR manually. Candidates: {existing}"
    )

def find_one_csv(directory, split):
    directory = Path(directory)
    patterns = {
        "train": ["*train*.csv"],
        "val": ["*val*.csv", "*validation*.csv"],
        "test": ["*test*.csv"],
    }[split]

    found = []
    for pat in patterns:
        found.extend(directory.glob(pat))

    found = sorted(set(
        p for p in found
        if all(token not in p.name.lower()
               for token in ["prediction", "metric", "history", "condition"])
    ))

    if len(found) != 1:
        raise RuntimeError(
            f"Expected exactly one {split} split CSV under {directory}, "
            f"found {len(found)}: {[p.name for p in found]}"
        )
    return found[0]

SOURCE_MANIFEST_DIR = choose_manifest_dir()
train_csv = find_one_csv(SOURCE_MANIFEST_DIR, "train")
val_csv = find_one_csv(SOURCE_MANIFEST_DIR, "val")
test_csv = find_one_csv(SOURCE_MANIFEST_DIR, "test")

print("Using exact archived manifests from:", SOURCE_MANIFEST_DIR)
print("Train:", train_csv.name)
print("Val:  ", val_csv.name)
print("Test: ", test_csv.name)

In [ ]:
train_df = pd.read_csv(train_csv).reset_index(drop=True)
val_df = pd.read_csv(val_csv).reset_index(drop=True)
test_df = pd.read_csv(test_csv).reset_index(drop=True)

splits = {"train": train_df, "val": val_df, "test": test_df}

for split_name, frame in splits.items():
    missing = {PATH_COL, LABEL_COL, CONDITION_COL} - set(frame.columns)
    if missing:
        raise ValueError(f"{split_name} manifest is missing columns: {missing}")

    expected_n = EXPECTED_SPLIT_ROWS[split_name]
    assert len(frame) == expected_n, (
        f"{split_name}: expected {expected_n} rows from the final run, "
        f"got {len(frame)}"
    )

    missing_paths = (~frame[PATH_COL].map(lambda x: Path(x).is_file())).sum()
    if missing_paths:
        raise FileNotFoundError(
            f"{split_name}: {missing_paths} video paths do not currently exist. "
            "Update the path column or mount the dataset before reproducing features."
        )

print("Split sizes:", {k: len(v) for k, v in splits.items()})
print("\nTrain labels:\n", train_df[LABEL_COL].value_counts().sort_index())
print("\nValidation labels:\n", val_df[LABEL_COL].value_counts().sort_index())
print("\nTest labels:\n", test_df[LABEL_COL].value_counts().sort_index())
print("\nTest conditions:\n", test_df[CONDITION_COL].value_counts())

In [ ]:
# Validate clip-group separation where the archived manifests contain clip_group.
if all(GROUP_COL in frame.columns for frame in splits.values()):
    train_groups = set(train_df[GROUP_COL].astype(str))
    val_groups = set(val_df[GROUP_COL].astype(str))
    test_groups = set(test_df[GROUP_COL].astype(str))

    overlaps = {
        "train_val": len(train_groups & val_groups),
        "train_test": len(train_groups & test_groups),
        "val_test": len(val_groups & test_groups),
    }
    print("Group overlaps:", overlaps)
    assert overlaps == {"train_val": 0, "train_test": 0, "val_test": 0}
else:
    print("clip_group is not present in every saved split; overlap cannot be re-audited here.")

In [ ]:
# Copy the exact split manifests into the reproduction run folder.
for split_name, src in [("train", train_csv), ("val", val_csv), ("test", test_csv)]:
    shutil.copy2(src, MANIFEST_OUT_DIR / f"{split_name}.csv")
print("Copied exact split manifests to:", MANIFEST_OUT_DIR)

## 4. Visual dataset and uniform frame sampling

Frames are decoded with OpenCV, converted BGR → RGB, resized to 224×224,
converted to float tensors in `[0,1]`, and normalised with ImageNet mean and
standard deviation.

`NUM_FRAMES` positions are sampled uniformly over the complete video. If a
decoder returns fewer frames than requested, the last successfully decoded
frame is repeated.

In [ ]:
class VideoFrameDataset(Dataset):
    def __init__(self, frame, num_frames, image_size):
        self.df = frame.reset_index(drop=True)
        self.num_frames = int(num_frames)
        self.image_size = int(image_size)
        self.mean = torch.tensor(
            [0.485, 0.456, 0.406], dtype=torch.float32
        ).view(3, 1, 1)
        self.std = torch.tensor(
            [0.229, 0.224, 0.225], dtype=torch.float32
        ).view(3, 1, 1)

    def __len__(self):
        return len(self.df)

    def _preprocess(self, frame_bgr):
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame_rgb = cv2.resize(
            frame_rgb,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_AREA,
        )
        x = torch.from_numpy(frame_rgb).permute(2, 0, 1).float().div_(255.0)
        return (x - self.mean) / self.std

    def _sample_frames(self, path):
        cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            raise RuntimeError(f"Could not open video: {path}")

        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            raise RuntimeError(f"Video has no reported frames: {path}")

        wanted = np.linspace(0, total - 1, self.num_frames).astype(np.int64)
        wanted_set = set(wanted.tolist())

        frames = []
        i = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if i in wanted_set:
                frames.append(self._preprocess(frame))
                if len(frames) == self.num_frames:
                    break
            i += 1

        cap.release()

        if not frames:
            raise RuntimeError(f"No frames decoded from: {path}")

        while len(frames) < self.num_frames:
            frames.append(frames[-1].clone())

        return torch.stack(frames, dim=0)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        frames = self._sample_frames(row[PATH_COL])
        label = int(row[LABEL_COL])
        return (
            frames,
            torch.tensor(label, dtype=torch.long),
            torch.tensor(idx, dtype=torch.long),
        )

## 5. Frozen ConvNeXt-Base feature extractor

The ImageNet classifier layer is replaced with identity so each frame produces a
1024-D representation. Frame representations are mean-pooled to one clip vector.

The encoder is frozen. Only the later linear classifier is trained.

In [ ]:
class ConvNeXtBaseClipFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        weights = ConvNeXt_Base_Weights.IMAGENET1K_V1
        self.backbone = convnext_base(weights=weights)
        self.backbone.classifier[2] = nn.Identity()
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, frames):
        # frames: [B, T, C, H, W]
        b, t, c, h, w = frames.shape
        flat = frames.reshape(b * t, c, h, w)
        features = self.backbone(flat)             # [B*T, 1024]
        features = features.reshape(b, t, -1)      # [B, T, 1024]
        return features.mean(dim=1)                # [B, 1024]

feature_extractor = ConvNeXtBaseClipFeatures().to(DEVICE).eval()
print("Trainable encoder parameters:",
      sum(p.numel() for p in feature_extractor.parameters() if p.requires_grad))

## 6. Feature-cache extraction

The expensive part is video decoding plus ConvNeXt inference. Features are cached
once per split so the classifier can be trained repeatedly without decoding videos.

The cache stores:

- one 1024-D vector per clip;
- the binary label;
- the original split-row index.

If a complete cache already exists, the extraction is skipped.

In [ ]:
def make_feature_loader(frame):
    ds = VideoFrameDataset(frame, NUM_FRAMES, IMAGE_SIZE)
    kwargs = dict(
        dataset=ds,
        batch_size=FEATURE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=False,
    )
    if NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
        kwargs["timeout"] = 240
    return DataLoader(**kwargs)

@torch.inference_mode()
def extract_split_features(split_name, frame, overwrite=False):
    cache_path = FEATURE_DIR / f"{split_name}_clip_features.pt"

    if cache_path.exists() and not overwrite:
        payload = torch.load(cache_path, map_location="cpu")
        if len(payload["y"]) == len(frame):
            print(f"Using existing {split_name} cache:", cache_path)
            return payload
        print("Existing cache row count does not match; regenerating.")

    loader = make_feature_loader(frame)
    xs, ys, idxs = [], [], []

    feature_extractor.eval()
    for frames, labels, indices in tqdm(loader, desc=f"Extract {split_name}"):
        frames = frames.to(DEVICE, non_blocking=False)
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            features = feature_extractor(frames)

        xs.append(features.float().cpu())
        ys.append(labels.cpu())
        idxs.append(indices.cpu())

    payload = {
        "X": torch.cat(xs, dim=0),
        "y": torch.cat(ys, dim=0),
        "idx": torch.cat(idxs, dim=0),
        "num_frames": NUM_FRAMES,
        "feature_dim": FEATURE_DIM,
    }

    assert payload["X"].shape == (len(frame), FEATURE_DIM)
    assert len(payload["y"]) == len(frame)

    torch.save(payload, cache_path)
    print("Saved:", cache_path, tuple(payload["X"].shape))
    return payload

train_cache = extract_split_features("train", train_df)
val_cache = extract_split_features("val", val_df)
test_cache = extract_split_features("test", test_df)

## 7. Sanity-check cached features

In [ ]:
for split_name, payload in [
    ("train", train_cache),
    ("val", val_cache),
    ("test", test_cache),
]:
    X = payload["X"]
    print(
        split_name,
        "shape=", tuple(X.shape),
        "mean=", float(X.mean()),
        "std=", float(X.std()),
        "finite=", bool(torch.isfinite(X).all()),
    )
    assert torch.isfinite(X).all()
    assert X.std() > 0

## 8. Linear classifier

Final head:

```text
LayerNorm(1024)
Linear(1024 -> 2)
```

It is trained for 30 epochs with AdamW. The checkpoint with the highest
**validation F1** is retained.

In [ ]:
X_train, y_train = train_cache["X"].float(), train_cache["y"].long()
X_val, y_val = val_cache["X"].float(), val_cache["y"].long()
X_test, y_test = test_cache["X"].float(), test_cache["y"].long()

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=CLASSIFIER_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=CLASSIFIER_BATCH_SIZE,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=CLASSIFIER_BATCH_SIZE,
    shuffle=False,
)

class ClipLinearClassifier(nn.Module):
    def __init__(self, dim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, 2),
        )

    def forward(self, x):
        return self.net(x)

classifier = ClipLinearClassifier(FEATURE_DIM).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=CLASSIFIER_LR,
    weight_decay=CLASSIFIER_WEIGHT_DECAY,
)

classifier

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    y_true, y_pred, y_prob = [], [], []

    for X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(X)
        loss = criterion(logits, y)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(dim=1)

        total_loss += loss.item() * len(y)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
    }
    return metrics, y_true, y_pred, y_prob

In [ ]:
best_val_f1 = -np.inf
history = []
checkpoint_path = CHECKPOINT_DIR / "best_classifier.pt"

for epoch in range(1, CLASSIFIER_EPOCHS + 1):
    classifier.train()
    train_loss = 0.0
    train_true, train_pred = [], []

    for X, y in train_loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = classifier(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        pred = logits.argmax(dim=1)
        train_loss += loss.item() * len(y)
        train_true.extend(y.detach().cpu().numpy())
        train_pred.extend(pred.detach().cpu().numpy())

    train_metrics = {
        "loss": train_loss / len(train_loader.dataset),
        "accuracy": accuracy_score(train_true, train_pred),
        "balanced_accuracy": balanced_accuracy_score(train_true, train_pred),
        "f1": f1_score(train_true, train_pred, zero_division=0),
        "precision": precision_score(train_true, train_pred, zero_division=0),
        "recall": recall_score(train_true, train_pred, zero_division=0),
    }

    val_metrics, _, _, _ = evaluate(classifier, val_loader)

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)
    pd.DataFrame(history).to_csv(RESULT_DIR / "training_history.csv", index=False)

    print(
        f"epoch={epoch:02d} "
        f"train_f1={train_metrics['f1']:.4f} "
        f"val_f1={val_metrics['f1']:.4f} "
        f"val_auc={val_metrics['auc']:.4f}"
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        torch.save(
            {
                "model_state_dict": classifier.state_dict(),
                "epoch": epoch,
                "val_metrics": val_metrics,
                "feature_dim": FEATURE_DIM,
                "num_frames": NUM_FRAMES,
                "seed": SEED,
            },
            checkpoint_path,
        )

print("Best validation F1:", best_val_f1)
print("Checkpoint:", checkpoint_path)

## 9. Final held-out test evaluation

In [ ]:
ckpt = torch.load(checkpoint_path, map_location=DEVICE)
classifier.load_state_dict(ckpt["model_state_dict"])

test_metrics, y_true, y_pred, y_prob = evaluate(classifier, test_loader)
display(pd.DataFrame([test_metrics]))
pd.DataFrame([test_metrics]).to_csv(RESULT_DIR / "test_metrics.csv", index=False)

test_predictions = test_df.copy()
test_predictions["y_true"] = y_true
test_predictions["y_pred"] = y_pred
test_predictions["fake_probability"] = y_prob
test_predictions.to_csv(RESULT_DIR / "test_predictions.csv", index=False)

## 10. Condition-level behaviour

In [ ]:
rows = []
for condition, group in test_predictions.groupby(CONDITION_COL):
    true_values = sorted(group["y_true"].unique().tolist())
    pred_fake_rate = float(group["y_pred"].mean())
    mean_prob = float(group["fake_probability"].mean())

    if len(true_values) == 1 and int(true_values[0]) == 0:
        success = 1.0 - pred_fake_rate
        interpretation = "real specificity"
    elif len(true_values) == 1 and int(true_values[0]) == 1:
        success = pred_fake_rate
        interpretation = "fake recall"
    else:
        success = float((group["y_true"] == group["y_pred"]).mean())
        interpretation = "accuracy"

    rows.append({
        "condition": condition,
        "n": len(group),
        "true_labels": str(true_values),
        "condition_success_rate": success,
        "interpretation": interpretation,
        "predicted_fake_rate": pred_fake_rate,
        "mean_fake_probability": mean_prob,
    })

condition_metrics = pd.DataFrame(rows).sort_values("condition")
display(condition_metrics)
condition_metrics.to_csv(RESULT_DIR / "condition_metrics.csv", index=False)

Expected historical test condition counts/results, used only as a comparison
check after rerunning:

| condition | n | final success rate |
|---|---:|---:|
| `fake_video_fake_audio` | 1922 | 0.910510 fake recall |
| `fake_video_real_audio` | 1981 | 0.911661 fake recall |
| `real` | 3084 | 0.452010 specificity |
| `real_video_fake_audio` | 1966 | 0.920651 fake recall |

## 11. Plots

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred,
    display_labels=["Real", "Fake"],
    values_format="d",
    ax=ax,
)
ax.set_title(f"{RUN_NAME}: test confusion matrix")
fig.tight_layout()
fig.savefig(PLOT_DIR / "confusion_matrix.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_true, y_prob, ax=ax)
ax.set_title(f"{RUN_NAME}: test ROC")
fig.tight_layout()
fig.savefig(PLOT_DIR / "roc_curve.png", dpi=200)
plt.show()

hist = pd.read_csv(RESULT_DIR / "training_history.csv")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(hist["epoch"], hist["train_f1"], label="train_f1")
ax.plot(hist["epoch"], hist["val_f1"], label="val_f1")
ax.set_xlabel("Epoch")
ax.set_ylabel("F1")
ax.set_title(f"{RUN_NAME}: classifier training")
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "training_f1.png", dpi=200)
plt.show()

## 12. Compare the reproduction with the archived final metrics

This is an **audit**, not a mechanism that changes the reproduced result.
The historical values remain the source of truth for the dissertation. A close
rerun supports reproducibility; a difference should be investigated rather than
silently replacing the archived values.

In [ ]:
comparison_rows = []
for metric, historical_value in EXPECTED_HISTORICAL_METRICS.items():
    reproduced_value = float(test_metrics[metric])
    comparison_rows.append({
        "metric": metric,
        "historical_final": historical_value,
        "reproduced": reproduced_value,
        "absolute_difference": abs(reproduced_value - historical_value),
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)
comparison.to_csv(RESULT_DIR / "historical_vs_reproduced_metrics.csv", index=False)

## 13. Archive the reproduction run

Large cached feature tensors are excluded from the ZIP by default. They can be
regenerated from the exact manifests and raw dataset.

In [ ]:
def build_archive_index():
    rows = []
    for p in RUN_DIR.rglob("*"):
        if p.is_file():
            rows.append({
                "relative_path": p.relative_to(RUN_DIR).as_posix(),
                "size_mb": p.stat().st_size / 1024**2,
            })
    out = pd.DataFrame(rows).sort_values("relative_path")
    out.to_csv(RUN_DIR / "archive_index.csv", index=False)
    return out

display(build_archive_index())

EXPORT_DIR = PROJECT_ROOT / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = EXPORT_DIR / f"{RUN_NAME}_{stamp}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RUN_DIR.rglob("*"):
        if not p.is_file():
            continue
        rel = p.relative_to(RUN_DIR)
        if "features" in rel.parts and p.suffix == ".pt":
            continue
        zf.write(p, arcname=(RUN_DIR.name / rel) if False else str(Path(RUN_DIR.name) / rel))

print("Created:", zip_path)
display(FileLink(str(zip_path)))

## 14. Interpretation reminder

This experiment is a **visual-only binary any-fake baseline**.

A correct binary prediction on `real_video_fake_audio` is not proof that the
visual model detected synthetic audio. Because the visual stream is semantically
real in that condition, high fake prediction rates are compatible with visual or
production-correlated shortcut cues.

Also remember that the in-domain split is **clip-group-disjoint**, not guaranteed
identity-disjoint.